In [1]:
import pandas as pd 
df= pd.read_csv("combined30k.csv")
df

,url,label,source
0,https://official-website.ghost.io/en-start/,1,phishtank
1,http://allegrolokalnie.oferta6311957.click,1,phishtank
2,https://allegro.45689459868945.sbs/,1,phishtank
3,https://allegro.45689459868945.sbs,1,phishtank
4,https://allegrolokalnie.l23arg45el4.click,1,phishtank
...,...,...,...
167162,https://cienciaempresarial.com.br,0,majestic
167163,https://kleingarten-bund.de,0,majestic
167164,https://mytimeactive.co.uk,0,majestic
167165,https://casinogoodwins.com,0,majestic


In [ ]:

print(df.isnull().sum())

print(df.duplicated().sum())

url        4
label      0
source    96
dtype: int64
13


In [4]:
df.loc[74364:74463, "source"] = "openphish"


In [5]:
print(df.isnull().sum())

url       4
label     0
source    0
dtype: int64


In [3]:
rows_with_nulls = df[df.isnull().any(axis=1)]
rows_with_nulls

,url,label,source
74364,esxcc.com/js/index.htm?us.battle.net/noghn/en/...,1,NaN
74365,wwweira&nvinipncHwV%yDaH/yEu\n6(rTu =g0m...,1,NaN
74366,'www.institutocgr.coo/web/media/syqvem/dk-ij!...,1,NaN
74367,Y koջΧDlqt/;,1,NaN
74369,ruta89fm.com/images/AS@Vies/1i75cf7b16vc<Fd16...,1,NaN
...,...,...,...
74459,r $cY,1,NaN
74460,D +ZIdN -lh9LhDKh%Yqñd,1,NaN
74461,"އ~\(ǽJ$Xm 5{kC_QʩBǆc2#S_^߱ k\m-""i!LGbb...",1,NaN
74462,"luB'JIGn""(0",1,NaN


In [6]:
df = df.dropna()

df = df.drop_duplicates()

df = df.reset_index(drop=True)

In [ ]:

print(df.isnull().sum())

print(df.duplicated().sum())

url       0
label     0
source    0
dtype: int64
0


In [ ]:
import pandas as pd
import tldextract
import re
from urllib.parse import urlparse
from math import log2

def extract_url_features(url):
    features = {}
  
    features['url_length'] = len(url)
    features['hostname_length'] = len(urlparse(url).netloc)
    features['count_dots'] = url.count('.')
    features['count_hyphens'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question'] = url.count('?')
    features['count_equals'] = url.count('=')
    features['count_digits'] = sum(c.isdigit() for c in url)
    features['count_special_chars'] = sum(not c.isalnum() for c in url)
    
   
    features['use_ip'] = 1 if re.match(r'http[s]?://\d+\.\d+\.\d+\.\d+', url) else 0
    
    features['https'] = 1 if url.startswith('https') else 0
    
   
    short_domains = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co']
    features['shortened_url'] = 1 if any(sd in url for sd in short_domains) else 0
    
   
    suspicious_words = ['login','secure','bank','account','update','verify']
    features['suspicious_words'] = 1 if any(word in url.lower() for word in suspicious_words) else 0
    
 
    features['num_subdirectories'] = urlparse(url).path.count('/')
    
    ext = tldextract.extract(url)
    features['tld_type'] = ext.suffix
    
    
    probs = [url.count(c)/len(url) for c in set(url)]
    features['entropy'] = -sum(p*log2(p) for p in probs)
    
    return features

df_features = df['url'].apply(extract_url_features).apply(pd.Series)

df_ml = pd.concat([df_features, df['label']], axis=1)


In [10]:
df_tot=pd.concat([df["url"],df_ml], axis=1)

In [11]:
df_tot

,url,url_length,hostname_length,count_dots,count_hyphens,count_at,count_question,count_equals,count_digits,count_special_chars,use_ip,https,shortened_url,suspicious_words,num_subdirectories,tld_type,entropy,label
0,https://official-website.ghost.io/en-start/,43,25,2,2,0,0,0,0,9,0,1,0,0,2,io,4.053717,1
1,http://allegrolokalnie.oferta6311957.click,42,35,2,0,0,0,0,7,5,0,0,0,0,0,click,4.329718,1
2,https://allegro.45689459868945.sbs/,35,26,2,0,0,0,0,14,6,0,1,0,0,1,sbs,4.085588,1
3,https://allegro.45689459868945.sbs,34,26,2,0,0,0,0,14,5,0,1,0,0,0,sbs,4.094097,1
4,https://allegrolokalnie.l23arg45el4.click,41,33,2,0,0,0,0,5,5,0,1,0,0,0,click,4.158497,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167148,https://cienciaempresarial.com.br,33,25,2,0,0,0,0,0,5,0,1,0,0,0,com.br,3.960320,0
167149,https://kleingarten-bund.de,27,19,1,1,0,0,0,0,5,0,1,0,0,0,de,4.078419,0
167150,https://mytimeactive.co.uk,26,18,2,0,0,0,0,0,5,0,1,0,0,0,co.uk,3.931209,0
167151,https://casinogoodwins.com,26,18,1,0,0,0,0,0,4,0,1,0,0,0,com,3.825252,0


In [12]:
print(df_tot.isnull().sum())

url                    0
url_length             0
hostname_length        0
count_dots             0
count_hyphens          0
count_at               0
count_question         0
count_equals           0
count_digits           0
count_special_chars    0
use_ip                 0
https                  0
shortened_url          0
suspicious_words       0
num_subdirectories     0
tld_type               0
entropy                0
label                  0
dtype: int64


In [ ]:
# encoding tld types

df_tot = pd.get_dummies(df_tot, columns=['tld_type'])
df_tot

,url,url_length,hostname_length,count_dots,count_hyphens,count_at,count_question,count_equals,count_digits,count_special_chars,...,tld_type_wv.us,tld_type_xn--fiqs8s,tld_type_xn--p1ai,tld_type_xxx,tld_type_xyz,tld_type_yachts,tld_type_yk.ca,tld_type_yt,tld_type_zone,tld_type_zp.ua
0,https://official-website.ghost.io/en-start/,43,25,2,2,0,0,0,0,9,...,False,False,False,False,False,False,False,False,False,False
1,http://allegrolokalnie.oferta6311957.click,42,35,2,0,0,0,0,7,5,...,False,False,False,False,False,False,False,False,False,False
2,https://allegro.45689459868945.sbs/,35,26,2,0,0,0,0,14,6,...,False,False,False,False,False,False,False,False,False,False
3,https://allegro.45689459868945.sbs,34,26,2,0,0,0,0,14,5,...,False,False,False,False,False,False,False,False,False,False
4,https://allegrolokalnie.l23arg45el4.click,41,33,2,0,0,0,0,5,5,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167148,https://cienciaempresarial.com.br,33,25,2,0,0,0,0,0,5,...,False,False,False,False,False,False,False,False,False,False
167149,https://kleingarten-bund.de,27,19,1,1,0,0,0,0,5,...,False,False,False,False,False,False,False,False,False,False
167150,https://mytimeactive.co.uk,26,18,2,0,0,0,0,0,5,...,False,False,False,False,False,False,False,False,False,False
167151,https://casinogoodwins.com,26,18,1,0,0,0,0,0,4,...,False,False,False,False,False,False,False,False,False,False


In [15]:
df_tot.columns

Index(['url', 'url_length', 'hostname_length', 'count_dots', 'count_hyphens',
       'count_at', 'count_question', 'count_equals', 'count_digits',
       'count_special_chars',
       ...
       'tld_type_wv.us', 'tld_type_xn--fiqs8s', 'tld_type_xn--p1ai',
       'tld_type_xxx', 'tld_type_xyz', 'tld_type_yachts', 'tld_type_yk.ca',
       'tld_type_yt', 'tld_type_zone', 'tld_type_zp.ua'],
      dtype='object', length=822)

In [16]:
X = df_tot.drop('label', axis=1)  # Features
y = df_tot['label']               

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)


In [ ]:
# now , splitting the url
if 'url' in X_train.columns:
    X_train = X_train.drop('url', axis=1)
if 'url' in X_test.columns:
    X_test = X_test.drop('url', axis=1)


In [24]:
print(X_train.dtypes)


url_length         int64
hostname_length    int64
count_dots         int64
count_hyphens      int64
count_at           int64
                   ...  
tld_type_yachts     bool
tld_type_yk.ca      bool
tld_type_yt         bool
tld_type_zone       bool
tld_type_zp.ua      bool
Length: 820, dtype: object


In [ ]:
# any remaining object columns to numeric 
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# Fill any NaN values
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import pandas as pd
import numpy as np


In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=100,       # Number of trees
    random_state=42,        # For reproducibility
    class_weight='balanced' # Handles class imbalance
)

rf_model.fit(X_train, y_train)

print("Model training complete!")


Model training complete!


In [ ]:
# using RF on the test set
y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9987137686578326
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3000
           1       1.00      1.00      1.00     30431

    accuracy                           1.00     33431
   macro avg       0.99      1.00      1.00     33431
weighted avg       1.00      1.00      1.00     33431

Confusion Matrix:
 [[ 2991     9]
 [   34 30397]]


In [ ]:
# Save model
joblib.dump(rf_model, "url_fraud_detector.pkl")

joblib.dump(X_train.columns.tolist(), "feature_columns.pkl")

print("Model and feature columns saved successfully!")


Model and feature columns saved successfully!


In [ ]:
# Example: function to extract URL features (from before)
def extract_url_features(url):
    import re
    import tldextract
    from urllib.parse import urlparse
    from math import log2
    
    features = {}
    features['url_length'] = len(url)
    features['hostname_length'] = len(urlparse(url).netloc)
    features['count_dots'] = url.count('.')
    features['count_hyphens'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question'] = url.count('?')
    features['count_equals'] = url.count('=')
    features['count_digits'] = sum(c.isdigit() for c in url)
    features['count_special_chars'] = sum(not c.isalnum() for c in url)
    features['use_ip'] = 1 if re.match(r'http[s]?://\d+\.\d+\.\d+\.\d+', url) else 0
    features['https'] = 1 if url.startswith('https') else 0
    short_domains = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co']
    features['shortened_url'] = 1 if any(sd in url for sd in short_domains) else 0
    suspicious_words = ['login','secure','bank','account','update','verify']
    features['suspicious_words'] = 1 if any(word in url.lower() for word in suspicious_words) else 0
    features['num_subdirectories'] = urlparse(url).path.count('/')
    ext = tldextract.extract(url)
    features['tld_type'] = ext.suffix
    probs = [url.count(c)/len(url) for c in set(url)]
    features['entropy'] = -sum(p*log2(p) for p in probs)
    return features


new_url = "https://login.microsoftonline.com"

new_features = extract_url_features(new_url)
new_df = pd.DataFrame([new_features])

new_df = pd.get_dummies(new_df, columns=['tld_type'])


for col in X_train.columns:
    if col not in new_df.columns:
        new_df[col] = 0
new_df = new_df[X_train.columns]

prediction = rf_model.predict(new_df)[0]

if prediction == 1:
    print(f"The URL '{new_url}' is likely FRAUD.")
else:
    print(f"The URL '{new_url}' is SAFE.")


The URL 'https://login.microsoftonline.com' is likely FRAUD.


/tmp/ipykernel_104748/2250999447.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[col] = 0
/tmp/ipykernel_104748/2250999447.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[col] = 0
/tmp/ipykernel_104748/2250999447.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[col]